# 🚀 HỆ THỐNG GIÁM SÁT HÀNH VI VIDEO THỜI GIAN THỰC & KIỂM SOÁT ĐA LUỒNG
> **Học phần**: Hệ Điều Hành - GVHD: Thầy Nguyễn Tấn Duẩn  
> **Đề tài**: *Hệ thống giám sát hành vi video thời gian thực & Phân tích bằng AI kết hợp kiểm soát đa luồng*  
> **Môi trường**: Điện toán đám mây Google Colab (GPU NVIDIA Tesla T4 16GB VRAM - Miễn phí 100%)

---

### 📌 Hướng dẫn 2 bước chạy 1-Click (Dành cho Thành Viên Nhóm & Thầy Cô):
1. **Bật GPU T4**: Vào menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU** (Đã được cấu hình tự động sẵn).
2. **Chạy hệ thống**: Bấm phím tắt **`Ctrl + F9`** (hoặc menu **Runtime** $\rightarrow$ **Run all**).
3. **Lấy link Web**: Đợi khoảng 45 giây, cuộn xuống ô cuối cùng để bấm vào đường link màu xanh `https://xxxx.trycloudflare.com` và sử dụng web!

In [ ]:
# [BƯỚC 1] Kiểm tra card đồ họa rời NVIDIA Tesla T4 do Google cấp
!nvidia-smi

In [ ]:
# [BƯỚC 2] Tải mã nguồn mới nhất từ GitHub & Cài đặt thư viện
import os
import sys

# Clone repo nếu chưa có, hoặc pull bản mới nhất nếu đã có
if not os.path.exists("os-multithreaded-video-analytics"):
    !git clone https://github.com/KhoaK50/os-multithreaded-video-analytics.git
    %cd os-multithreaded-video-analytics
else:
    %cd os-multithreaded-video-analytics
    !git pull origin main

# Cài đặt các thư viện cần thiết
!pip install -q -r requirements.txt
print("[+] Đã cài đặt hoàn tất toàn bộ thư viện!")

In [ ]:
# [BƯỚC 3] Thiết lập đường truyền bảo mật Cloudflare & Khởi chạy Web Server
import subprocess
import time
import re
import threading

# Cài đặt Cloudflare Tunnel cho Linux (chạy nền miễn phí, không cần đăng ký tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i -E cloudflared-linux-amd64.deb > /dev/null 2>&1

print("=" * 72)
print("[*] Đang thiết lập đường truyền bảo mật Cloudflare Tunnel...")
print("=" * 72)

# Chạy cloudflared tunnel trỏ tới cổng 8000
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Quét tìm đường link trycloudflare.com công khai
public_url = None
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "*" * 72)
    print("  🎉 HỆ THỐNG ĐÃ SẴN SÀNG TRÊN GPU NVIDIA TESLA T4 CỦA GOOGLE!")
    print(f"  🔗 BẤM VÀO ĐÂY ĐỂ MỞ WEB APP: {public_url}")
    print("*" * 72 + "\n")
else:
    print("[!] Không bắt được URL Cloudflare tự động. Vui lòng kiểm tra lại log.")

# Khởi chạy máy chủ FastAPI của dự án
!python main.py --no-browser --port 8000
